# RV32I-MLA Bring-up Notebook (PYNQ)

This notebook is split into small, focused cells for BRAM and mailbox bring-up.

**Recommended order**
1. Load the overlay
2. Inspect the memory map
3. Create the MMIO handle
4. Run PS write/read sanity first
5. Then run mailbox tests
6. Use the liveness scan only if mailbox behavior is still unclear


## 1) Imports and basic configuration

Set the bitstream filename here.  
The matching `.hwh` file should be in the same directory and use the same basename.


In [ ]:
from pynq import Overlay, MMIO
import time

BIT_NAME = "rv32i_mla_bd.bit"


## 2) Helper functions

These helpers keep later cells short and readable.


In [ ]:
def hr(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def whex(x):
    return f"0x{(x & 0xFFFFFFFF):08X}"

def mmio_write32(mmio, off, val):
    mmio.write(off, val & 0xFFFFFFFF)

def mmio_read32(mmio, off):
    return mmio.read(off) & 0xFFFFFFFF

def dump_words(mmio, start_word=0, count=16):
    for i in range(count):
        off = 4 * (start_word + i)
        print(f"{start_word + i:3d}  {whex(mmio_read32(mmio, off))}")

def get_base_and_size(info):
    base = info.get("base_address", info.get("phys_addr", 0))
    size = info.get("size", info.get("addr_range", 0))
    return base, size


## 3) Load the overlay

This programs the PL and loads the hardware handoff metadata.


In [ ]:
hr("Load Overlay")
ol = Overlay(BIT_NAME)

print(f"Loaded      : {BIT_NAME}")
print(f"IP count    : {len(ol.ip_dict)}")
print(f"Mem regions : {len(ol.mem_dict)}")


## 4) Inspect available memory regions

Confirm that `axi_bram` is present and note its base address and range.


In [ ]:
hr("Memory Map (mem_dict)")
for name, info in ol.mem_dict.items():
    base, size = get_base_and_size(info)
    print(f"- {name:20s} base={whex(base)} size={whex(size)}")


## 5) Select the BRAM region and create MMIO

This cell creates the MMIO object used by the rest of the notebook.


In [ ]:
if "axi_bram" not in ol.mem_dict:
    raise RuntimeError("mem_dict has no 'axi_bram'. Check BD naming and address map.")

bram_info = ol.mem_dict["axi_bram"]
BRAM_BASE, BRAM_SIZE = get_base_and_size(bram_info)

hr("Selecting BRAM Region")
print(f"axi_bram base = {whex(BRAM_BASE)}")
print(f"axi_bram size = {whex(BRAM_SIZE)}")

mmio = MMIO(BRAM_BASE, BRAM_SIZE)


## 6) Inspect initial BRAM contents

A quick look at the first few words before any tests.


In [ ]:
hr("Initial BRAM [0..15]")
dump_words(mmio, start_word=0, count=16)


## 7) PS write/read sanity check

This is the most important first test.

If PS writes do not read back correctly here, do **not** trust mailbox tests yet.


In [ ]:
hr("PS Write/Read Sanity")

tests = [
    (0x00, 0xAAAAAAAA),
    (0x04, 0x55555555),
    (0x08, 0x12345678),
    (0x0C, 0xDEADBEEF),
]

for off, val in tests:
    before = mmio_read32(mmio, off)
    mmio_write32(mmio, off, val)
    time.sleep(0.01)
    after = mmio_read32(mmio, off)
    print(f"off={whex(off)}  before={whex(before)}  wrote={whex(val)}  after={whex(after)}")

print("\nInterpretation:")
print("- If AFTER matches WROTE, PS <-> BRAM path is working.")
print("- If AFTER stays constant or wrong, fix BD / bitstream / MMIO path first.")


## 8) Mailbox constants

These offsets match the current mailbox convention:
- word 0: command
- word 1: status
- word 2: cycles


In [ ]:
CMD_OFF    = 0x00
STATUS_OFF = 0x04
CYCLES_OFF = 0x08


## 9) Clear mailbox words

Run this before mailbox testing to start from a clean state.


In [ ]:
hr("Clear Mailbox")
mmio_write32(mmio, CMD_OFF, 0)
mmio_write32(mmio, STATUS_OFF, 0)
mmio_write32(mmio, CYCLES_OFF, 0)
time.sleep(0.01)

print("CMD    =", whex(mmio_read32(mmio, CMD_OFF)))
print("STATUS =", whex(mmio_read32(mmio, STATUS_OFF)))
print("CYCLES =", whex(mmio_read32(mmio, CYCLES_OFF)))


## 10) Single mailbox trigger test

This writes `CMD = 1` and polls `STATUS` until it becomes `2` or times out.


In [ ]:
hr("Mailbox Test (CMD=1, poll STATUS)")

print("Triggering CPU with CMD = 1")
mmio_write32(mmio, CMD_OFF, 1)

timeout_s = 2.0
poll_s = 0.02
t0 = time.time()
last = None

while time.time() - t0 < timeout_s:
    st = mmio_read32(mmio, STATUS_OFF)
    if st != last:
        print("STATUS =", whex(st))
        last = st
    if st == 2:
        break
    time.sleep(poll_s)

if last == 2:
    print("✅ CPU DONE detected (STATUS=2)")
else:
    print("⚠️ Timeout waiting for CPU DONE (STATUS=2)")

print("\nMailbox at end:")
print("CMD    =", whex(mmio_read32(mmio, CMD_OFF)))
print("STATUS =", whex(mmio_read32(mmio, STATUS_OFF)))
print("CYCLES =", whex(mmio_read32(mmio, CYCLES_OFF)))


## 11) Clear command after completion

If your CPU retriggers while `CMD` stays high, use this cell after a successful run.


In [ ]:
hr("Clear Command After Run")
mmio_write32(mmio, CMD_OFF, 0)
time.sleep(0.01)

print("CMD    =", whex(mmio_read32(mmio, CMD_OFF)))
print("STATUS =", whex(mmio_read32(mmio, STATUS_OFF)))
print("CYCLES =", whex(mmio_read32(mmio, CYCLES_OFF)))


## 12) Re-trigger test

This helps confirm that success is repeatable and not a one-off.


In [ ]:
hr("Re-trigger CPU 3 times")

for run in range(3):
    print(f"\nRun #{run + 1}")

    mmio_write32(mmio, STATUS_OFF, 0)
    mmio_write32(mmio, CMD_OFF, 0)
    time.sleep(0.01)

    mmio_write32(mmio, CMD_OFF, 1)

    t0 = time.time()
    done = False
    while time.time() - t0 < 2.0:
        st = mmio_read32(mmio, STATUS_OFF)
        if st == 2:
            done = True
            break
        time.sleep(0.02)

    if done:
        cyc = mmio_read32(mmio, CYCLES_OFF)
        print("✅ DONE. cycles =", whex(cyc))
    else:
        print("⚠️ Timeout. STATUS =", whex(mmio_read32(mmio, STATUS_OFF)))

    mmio_write32(mmio, CMD_OFF, 0)
    time.sleep(0.01)


## 13) BRAM liveness scan

Use this only if you suspect the CPU is writing somewhere unexpected.
It samples a small BRAM window twice and reports any differences.


In [ ]:
hr("BRAM Liveness Scan")

start_word = 0
count_words = 64

snap1 = [mmio_read32(mmio, 4 * (start_word + i)) for i in range(count_words)]
time.sleep(0.1)
snap2 = [mmio_read32(mmio, 4 * (start_word + i)) for i in range(count_words)]

deltas = []
for i, (a, b) in enumerate(zip(snap1, snap2)):
    if a != b:
        deltas.append((start_word + i, a, b))

if not deltas:
    print("No deltas detected in sampled window.")
    print("If you expected CPU activity, check reset, clock, BRAM wiring, or mailbox protocol.")
else:
    print(f"Found {len(deltas)} changing words:")
    for (w, a, b) in deltas[:20]:
        print(f"word {w:4d}: {whex(a)} -> {whex(b)}")
    if len(deltas) > 20:
        print("... (truncated)")


## 14) Final notes

A healthy sequence usually looks like this:

1. PS write/read sanity passes
2. Mailbox clear reads back zeros
3. `CMD = 1` is written
4. `STATUS` eventually becomes `2`
5. `CMD` is cleared again if retriggering is level-sensitive
